# Quantum Funding Chart — Smooth World Edition (v3)

Fixed SVG path parser. Gray world silhouette background + geographic bubbles.

**Run all cells from the top.**

In [ ]:
from pathlib import Path
import os, subprocess

PROJECT_DIR = Path("/content/mit-prox-rd")
REPO_URL = "https://github.com/qimono-repos/mit-prox-rd.git"

if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", "trunk", "--single-branch", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", "trunk"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "-B", "trunk", "FETCH_HEAD"], check=True)

os.chdir(PROJECT_DIR)
print("Ready:", Path.cwd())

## 1. Imports and Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, PathPatch
from matplotlib.path import Path as MplPath
import numpy as np
from pathlib import Path

df = pd.read_csv("assets/quantum_funding_updated.csv", encoding="utf-8-sig")
print(f"Loaded {len(df)} records.")
display(df.head())

## 2. Configuration

In [ ]:
values = {
    "United States": 1006, "Canada": 38.5, "Brazil": 5.5,
    "United Kingdom": 330, "Germany": 848, "Netherlands": 85,
    "Switzerland": 23, "Austria": 24, "Italy": 36,
    "India": 90, "South Korea": 140, "Japan": 241,
    "Singapore": 44, "Australia": None, "China": 15000,
}

pos = {
    "Canada": (-105, 57), "United States": (-99, 37), "Brazil": (-52, -10),
    "United Kingdom": (-3, 54), "Netherlands": (5, 53), "Germany": (11, 51),
    "Switzerland": (8, 47), "Austria": (14, 47), "Italy": (12, 42),
    "India": (79, 22), "South Korea": (128, 36), "Japan": (139, 37),
    "Singapore": (104, 1.3), "Australia": (134, -25), "China": (105, 35),
}

## 3. Robust SVG World Layer

In [ ]:
from matplotlib.patches import PathPatch
from matplotlib.path import Path as MplPath
import xml.etree.ElementTree as ET
import re

def parse_svg_path(d):
    """Parse SVG absolute/relative M, C, L, H, V and Z commands into a Matplotlib Path.
    Supports repeated command arguments and scientific/decimal number formats.
    The source SVG in this project uses absolute M/C/Z commands, but this parser
    handles the common line commands too and won't mistake command letters for numbers.
    """
    token_re = re.compile(r"[A-Za-z]|[-+]?(?:\\d*\\.\\d+|\\d+\\.?\\d*)(?:[eE][-+]?\\d+)?")
    tokens = token_re.findall(d.replace(",", " "))
    verts, codes = [], []
    i, command = 0, None
    current = (0.0, 0.0)
    start = (0.0, 0.0)

    def number(index):
        if index >= len(tokens) or re.fullmatch(r"[A-Za-z]", tokens[index]):
            raise ValueError(f"Expected a numeric SVG coordinate near token {index}: {tokens[index:index+4]}")
        return float(tokens[index])

    def pair(index, relative=False):
        x, y = number(index), number(index+1)
        if relative:
            x += current[0]
            y += current[1]
        return (x, y)

    while i < len(tokens):
        token = tokens[i]
        if re.fullmatch(r"[A-Za-z]", token):
            command = token
            i += 1
            if command in "Zz":
                verts.append(start)
                codes.append(MplPath.CLOSEPOLY)
                current = start
                command = None
                continue
        if command is None:
            raise ValueError(f"SVG path data missing command near token {i}: {tokens[i:i+4]}")

        upper = command.upper()
        relative = command.islower()
        if upper == "M":
            point = pair(i, relative)
            verts.append(point)
            codes.append(MplPath.MOVETO)
            current = start = point
            i += 2
            # Additional coordinate pairs after M are implicitly L commands.
            command = "l" if relative else "L"
        elif upper == "L":
            point = pair(i, relative)
            verts.append(point)
            codes.append(MplPath.LINETO)
            current = point
            i += 2
        elif upper == "H":
            x = number(i) + (current[0] if relative else 0.0)
            current = (x, current[1])
            verts.append(current)
            codes.append(MplPath.LINETO)
            i += 1
        elif upper == "V":
            y = number(i) + (current[1] if relative else 0.0)
            current = (current[0], y)
            verts.append(current)
            codes.append(MplPath.LINETO)
            i += 1
        elif upper == "C":
            if i + 5 >= len(tokens):
                raise ValueError(f"Incomplete SVG cubic curve near token {i}: {tokens[i:]}")
            p1 = pair(i, relative)
            p2 = pair(i+2, relative)
            p3 = pair(i+4, relative)
            verts.extend([p1, p2, p3])
            codes.extend([MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4])
            current = p3
            i += 6
        else:
            raise ValueError(f"Unsupported SVG path command {command!r} near token {i}")

    if not verts:
        return [], []
    return verts, codes

def draw_world_svg(ax, svg_path="assets/world_silhouette_gray.svg"):
    """Draw smooth gray SVG world shapes into a Matplotlib axis."""
    tree = ET.parse(svg_path)
    root = tree.getroot()
    namespace = "{http://www.w3.org/2000/svg}"
    paths = root.findall(f".//{namespace}path") or root.findall(".//path")
    if not paths:
        raise ValueError(f"No SVG path elements found in {svg_path}")
    count = 0
    for element in paths:
        d = element.get("d")
        if not d:
            continue
        verts, codes = parse_svg_path(d)
        if not verts or codes[0] != MplPath.MOVETO:
            continue
        # SVG y grows downward, Matplotlib's geographic y grows upward. Flip y.
        viewbox = root.get("viewBox", "0 0 1000 500").split()
        vx, vy, vw, vh = map(float, viewbox)
        geo_verts = [((x-vx)/vw*360.0-180.0, 90.0-(y-vy)/vh*180.0) for x, y in verts]
        path = MplPath(geo_verts, codes)
        patch = PathPatch(path, facecolor="#e3e6e9", edgecolor="#c3cbd3",
                          linewidth=0.65, alpha=0.9, zorder=1, antialiased=True)
        ax.add_patch(patch)
        count += 1
    if count == 0:
        raise ValueError(f"SVG file parsed but no drawable land paths were found: {svg_path}")
    ax.set_xlim(-175, 185)
    ax.set_ylim(-60, 85)
    return ax


## 4. Generate Chart

In [ ]:
fig = plt.figure(figsize=(18, 11), dpi=150, facecolor="#fbfaf7")
ax = fig.add_axes([0.04, 0.16, 0.92, 0.66])
ax.set_facecolor("#fbfaf7")
ax.axis("off")

draw_world_svg(ax)

max_v = max(v for v in values.values() if v is not None and v < 5000)
for country, v in values.items():
    if country not in pos:
        continue
    x, y = pos[country]
    if v is None:
        ax.scatter([x], [y], s=80, facecolors="none", edgecolors="#6b7280", linewidths=1.5, zorder=4)
        continue
    if country == "China":
        radius, color, edge = 6.5, "#f6e8d4", "#b96b26"
    else:
        radius = 1.6 + 7.5 * np.sqrt(v / max_v)
        color, edge = "#69c7d7", "#00a9c0"
    ax.add_patch(Circle((x, y), radius=radius, facecolor=color, edgecolor=edge,
                        linewidth=1.3, alpha=0.8, zorder=3))
    if v >= 200 and country != "China":
        ax.text(x, y, f"{int(v)}", ha="center", va="center", fontsize=9, color="#17333a", weight="bold", zorder=5)
        ax.text(x, y + 4.5, country, ha="center", va="center", fontsize=8, color="#17333a", weight="bold", zorder=6)
    elif country == "China":
        ax.text(x, y, "~15k", ha="center", va="center", fontsize=9, color="#7d4519", weight="bold", zorder=5)
        ax.text(x, y + 5, "China\n(cum.)", ha="center", va="center", fontsize=7.5, color="#7d4519", weight="bold", zorder=6)

fig.text(0.04, 0.92, "Quantum funding — 2026 snapshot", fontsize=22, weight="bold", color="#1a1a1a")
fig.text(0.04, 0.875, "Gray world silhouette. Bubble area scales with annual / annualized figure (USD millions).", fontsize=11, color="#444")
fig.text(0.70, 0.20, "CHINA\n≈ US$15B cumulative\nhigh uncertainty", ha="center", va="center", fontsize=8.5, color="#7d4519",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#f6e8d4", edgecolor="#b96b26"))
fig.text(0.04, 0.05, "US = FY2024 enacted. See REFERENCES.md.", fontsize=8, color="#555")
plt.show()

## Notes

- Parser fixed in v3.
- China is a cumulative estimate (not on the same scale).